Innitially, we researched different APIs that can give us job listings and the specific information that we seeked to analyse. We settled on trying three different API: Adzuna, TheMuse, and finally Himalayas.

We first tried the Adzuna API, but while it did deliver us a lot of the information we needed, it did not deliver us the full description. When using the API, we discovered that the description gets truncated to only 300 characters, which is not enough for us to be able to extract the skills that are required for the job. After that, we saw that the API does return us a redirect url that contains the full description. However, when we checked it turned out that while the robots.txt does not stop us from parsing links of this type of path, the terms and conditions of Adzuna, do not allow us to directly scrap its contents. Hence, we has to give up on using that API. Below, we have shown the code that we used and would have used if we were allowed to parse the urls.

In [ ]:
# @title
import re
import os
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from google.colab import userdata

def clean_html(raw_text: str) -> str:
    """Removes raw HTML formatting tags and collapses extra whitespace."""
    if not raw_text:
        return ""
    clean_text = re.sub(r'<[^>]+>', ' ', raw_text)
    return " ".join(clean_text.split())

def extract_work_model_us(text: str) -> str:
    """Classifies US job listings into Remote, Hybrid, or On-site."""
    text_lower = text.lower()
    if "hybrid" in text_lower:
        return "Hybrid"
    elif any(k in text_lower for k in ["remote", "work from home", "telecommute", "wfh"]):
        return "Remote"
    return "On-site"

def url_reconstruction(url: str) -> float:
  """As we are not allowed to parse /land/ad/, we change the url to /details/id, which is not disallowed"""
    id = url.split('/')[-1]
    new_url = 'https://www.adzuna.com/details/' + id
    return new_url

def fetch_adzuna_us_jobs_robust(api_id: str, api_key: str, role: str = "data engineer", pages: int = 5) -> pd.DataFrame:
    records = []
    # Endpoint configured for US listings
    base_url = "https://api.adzuna.com/v1/api/jobs/us/search/"

    for page in range(1, pages + 1):
        params = {
            "app_id": api_id,
            "app_key": api_key,
            "results_per_page": 50,
            "what": role,
            "content-type": "application/json"
        }

        try:
            response = requests.get(f"{base_url}{page}", params=params, timeout=10)
            if response.status_code != 200:
                print(f"Skipping page {page}: API returned HTTP {response.status_code}")
                continue
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Network error on page {page}: {e}")
            continue

        for job in data.get("results", []):
            title = job.get("title") or ""
            raw_desc = job.get("description") or ""
            clean_desc = clean_html(raw_desc)
            combined_text = f"{title} {clean_desc}"
            company_info = job.get("company") or {}
            location_info = job.get("location") or {}
            sal_min = job.get("salary_min")
            sal_max = job.get("salary_max")
            avg_salary = (sal_min + sal_max) / 2 if (sal_min is not None and sal_max is not None) else None
            records.append({
                "job_id": job.get("id"),
                "title": clean_html(title),
                "company": company_info.get("display_name", "Unknown"),
                "city_state": location_info.get("display_name", "Unknown"),
                "area_hierarchy": location_info.get("area", []),
                "work_model": extract_work_model_us(combined_text),
                "salary_usd": avg_salary,
                "is_predicted_salary": job.get("salary_is_predicted"),
                "created_date": job.get("created"),
                "description_snippet": clean_desc,
                "redirect_url": url_reconstruction(job.get("redirect_url"))})

        time.sleep(0.5)

    df = pd.DataFrame(records)
    return df

APP_ID = userdata.get('APP_ID')
APP_KEY = userdata.get('APP_KEY')
df_us = fetch_adzuna_us_jobs_robust(APP_ID, APP_KEY, role="data science", pages=45)

# parsing the urls - not optimized as we never went through with it
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

rp = RobotFileParser()
robots_resp = requests.get("https://www.adzuna.com/robots.txt", headers=HEADERS, timeout=10)
if robots_resp.status_code == 200:
    rp.parse(robots_resp.text.splitlines())

session = requests.Session()
session.headers.update(HEADERS)


def fetch_full_text(url: str) -> str:
    """Fetches and extracts full job description text from an Adzuna details page."""
    if not url or pd.isna(url):
        return ""

    if not rp.can_fetch(HEADERS["User-Agent"], url):
        print(f"Skipped (disallowed): {url}")
        return ""

    try:
        time.sleep(1.5)
        response = session.get(url, timeout=10)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            container = soup.find("section", class_="adp-body mx-4 text-sm md:mx-0 md:text-base") # look only for the section containing the description
            target_element = container if container else soup.body

            if target_element:
                text = target_element.get_text(separator=" ", strip=True)
                return " ".join(text.split())

        elif response.status_code in (403, 429):
            print(f"Warning: HTTP {response.status_code} for {url}. Pausing 30s...")
            time.sleep(30)

    except requests.RequestException as err:
        print(f"Network error fetching {url}: {err}")

    return ""


df_us["full_description"] = ""
df_us["full_description"] = df_us["full_description"].fillna("").astype(str)
for idx in df_us.index:
    url = df_us.at[idx, "redirect_url"]
    df_us.at[idx, "full_description"] = fetch_full_text(url)
df_us.to_csv('adzune_data.csv', index=False)
print("Done! All rows processed and saved.")

After that we tried the Muse API, which did return us more than 3,000 job listings related to Data Science with their full description, and we were able to extract the skills that we needed. However, the problem here arose from the fact that the salary was within the plain text of the description. We couldn't easily extract it using regex because the salary format differed from listing to listing. Hence, we decided to use an LLM to extract the values we needed. However, after it was done, we were left with 585 listings only, which we felt was not enough for our analysis to be actually meaningful. Below, we have shown the code that we used for the extraction of data and calling the LLM.

In [ ]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup

def clean_html(html_text):
    if not html_text:
        return ""
    soup = BeautifulSoup(html_text, "html.parser")
    return soup.get_text(separator=" ").strip()

def get_segmented_muse_jobs(
    categories=["Data Science", "Data and Analytics", "Software Engineering"],
    levels=["Entry Level", "Mid Level", "Senior Level", "Management"],
    api_key=None):
    base_url = "https://www.themuse.com/api/public/jobs"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
    }

    seen_job_ids = set()
    all_jobs = []

    for cat in categories:
        for lvl in levels:
            print(f"\n--- Fetching: Category='{cat}' | Level='{lvl}' ---")

            page = 0
            # Cap each sub-query at page 99 to avoid hitting the 400 error
            max_allowed_page = 99

            while page <= max_allowed_page:
                params = {
                    "category": cat,
                    "level": lvl,
                    "page": page
                }
                if api_key:
                    params["api_key"] = api_key

                response = requests.get(base_url, headers=headers, params=params)

                # Stop this segment if we hit 400 or any non-200 status
                if response.status_code != 200:
                    print(f"  [Notice] Segment hit limit at page {page} (Status: {response.status_code}). Moving to next segment.")
                    break

                data = response.json()
                results = data.get("results", [])

                if not results:
                    break

                for item in results:
                    job_id = item.get("id")

                    # Deduplicate across categories/levels
                    if job_id in seen_job_ids:
                        continue
                    seen_job_ids.add(job_id)

                    raw_locations = [loc.get("name") for loc in item.get("locations", [])]
                    loc_str = ", ".join(raw_locations)

                    raw_html = item.get("contents", "")
                    plain_desc = clean_html(raw_html)

                    job_entry = {
                        "job_id": job_id,
                        "title": item.get("name"),
                        "company": item.get("company", {}).get("name"),
                        "locations": loc_str,
                        "categories": ", ".join([c.get("name") for c in item.get("categories", [])]),
                        "levels": ", ".join([l.get("name") for l in item.get("levels", [])]),
                        "publication_date": item.get("publication_date"),
                        "apply_url": item.get("refs", {}).get("landing_page"),
                        "description": plain_desc
                    }
                    all_jobs.append(job_entry)

                page_count = data.get("page_count", 0)
                if page >= page_count - 1:
                    break

                page += 1

            # Brief delay to prevent rate limit spikes
            time.sleep(0.2)

    return pd.DataFrame(all_jobs)

# Execute segmented data collection
df_muse_jobs = get_segmented_muse_jobs(
    categories=["Data Science", "Data and Analytics", "Software Engineering", "Computer and IT"],
    levels=["Entry Level", "Mid Level", "Senior Level", "Management"]
)

print(f"\nCompleted! Total unique jobs gathered: {len(df_muse_jobs)}")

In [ ]:
import os
import json
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

LUXIA_ENDPOINT = "https://bridge.luxiacloud.com/luxia/v1/chat"

SYSTEM_PROMPT = """You are an expert NLP data extraction system specializing in parsing salary information from unstructured job descriptions.

Your objective: Extract numerical salary bounds, currency, and payment frequency from the text.
Return ONLY valid JSON matching this exact structure with no Markdown wrappers, commentary, or explanations:

{
  "salary_found": boolean,     // true if an explicit dollar/currency figure or range is stated, false otherwise
  "salary_min": float or null, // lower numeric bound (e.g. 50.50 or 80000)
  "salary_max": float or null, // upper numeric bound (or same as min if exact figure)
  "currency": string or null,   // "USD", "EUR", "GBP", etc.
  "period": string or null      // MUST be one of: "HOURLY", "MONTHLY", "YEARLY"
}

If no numeric salary is mentioned, set "salary_found" to false and all other fields to null.
"""

API_KEY = userdata.get('LUXIA_API_KEY')   # fetch once, not per call
session = requests.Session()               # reuses connections
session.headers.update({"apikey": API_KEY, "Content-Type": "application/json"})

EMPTY = {"salary_found": False, "salary_min": None, "salary_max": None, "currency": None, "period": None}
KEYWORDS = ["$", "€", "£", "salary", "pay", "compensation", "/hr", "hourly", "annual"]
MAX_CHARS = 20000  # effectively no truncation; raise if your model's context allows

def normalize_to_annual_salary(extracted_data: dict) -> float:
    """Converts hourly/monthly rates to annualized figures and computes midpoint."""
    if not extracted_data.get("salary_found") or extracted_data.get("salary_min") is None:
        return None

    min_val = extracted_data["salary_min"]
    max_val = extracted_data["salary_max"] if extracted_data["salary_max"] is not None else min_val
    period = extracted_data.get("period", "YEARLY")

    # Midpoint estimation
    midpoint = (min_val + max_val) / 2.0

    # Multiplier conversion to Annualized Gross Salary
    if period == "HOURLY":
        annualized = midpoint * 2080  # 40 hrs/week * 52 weeks
    elif period == "MONTHLY":
        annualized = midpoint * 12
    else:  # YEARLY or default
        annualized = midpoint

    return round(annualized, 2)

def extract_salary_with_luxia(job_text: str, retries: int = 3) -> dict:
    if not isinstance(job_text, str) or not any(kw in job_text.lower() for kw in KEYWORDS):
        return EMPTY.copy()

    payload = {
        "model": "luxia3-llm-8b-0731",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Extract salary details from this job posting text:\n\n{job_text[:MAX_CHARS]}"}
        ],
        "temperature": 0.0,
        "max_tokens": 150,
    }

    for attempt in range(retries):
        try:
            r = session.post(LUXIA_ENDPOINT, json=payload, timeout=30)
            if r.status_code == 429:               # rate limited -> back off and retry
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            content = r.json()["choices"][0]["message"]["content"].strip()
            if content.startswith("```"):
                content = content.replace("```json", "").replace("```", "").strip()
            return json.loads(content)
        except Exception as e:
            if attempt == retries - 1:
                print(f"Extraction error: {e}")
            else:
                time.sleep(1 + attempt)
    return EMPTY.copy()

def run_extraction(df: pd.DataFrame, text_col: str = "description", workers: int = 12) -> pd.DataFrame:
    # Deduplicate identical descriptions so each is only sent once
    unique_texts = df[text_col].dropna().unique().tolist()
    results = {}

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(extract_salary_with_luxia, t): t for t in unique_texts}
        for f in tqdm(as_completed(futures), total=len(futures)):
            results[futures[f]] = f.result()

    extracted = df[text_col].map(lambda t: results.get(t, EMPTY.copy()))
    out = pd.json_normalize(extracted.tolist()).set_index(df.index)
    out["annual_salary"] = extracted.map(normalize_to_annual_salary)
    return pd.concat([df, out.add_prefix("llm_")], axis=1)

# Usage
df_muse_jobs = run_extraction(df, text_col="description", workers=7)

In [ ]:
HYBRID_RE = re.compile(
    r"hybrid (?:work|role|schedule|model|position|environment|setup|office)|"
    r"\bhybrid\b(?! cloud)|"
    r"\b[2-4]\s*(?:days?|x)\s*(?:per|a|/)\s*week\s*(?:in|at|on)[- ]?(?:the\s*)?(?:office|site)|"
    r"in[- ]office\s*[2-4]\s*days", re.I)
REMOTE_RE = re.compile(r"fully remote|100% remote|remote[- ]first|work from home|work remotely|this is a remote", re.I)

def muse_work_mode(location, description):
    loc, desc = str(location).lower(), str(description)
    if HYBRID_RE.search(desc):                 return "Hybrid"
    if "remote" in loc or "flexible" in loc:   return "Remote"
    if REMOTE_RE.search(desc):                 return "Remote"
    return "Onsite"

df_muse_jobs["work_mode"] = [muse_work_mode(l, d) for l, d in zip(df_muse_jobs["location"], df_muse_jobs["description"])]

In [ ]:
df_muse_jobs = pd.read_csv('/content/drive/MyDrive/muse_dataset_salaries_full (2).csv')

As we did not have a lot of remote jobs, we decided to also explore another data source, which was Himalayas API, which is a specialised API that has only remote job postings. This api was the easiest to work with as it both returned the full description as well as the salary without us needing to do any modifications.

In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import userdata

API_KEY = userdata.get('API_KEY')
BASE_URL = "https://himalayas.app/jobs/api/search"

HEADERS = {
    "User-Agent": "UniversityResearchProject/1.0",
    "Authorization": f"Bearer {API_KEY}",
}

SEARCH_QUERIES = [
    "data scientist",
    "machine learning",
    "data engineer",
    "data analyst",
    "artificial intelligence",
    "applied scientist",
]

jobs_by_guid = {}

for query in SEARCH_QUERIES:
    print(f"Searching for: '{query}'...")
    page = 1

    while True:
        params = {
            "q": query,
            "country": "US",       # Remove or change if you want global listings
            "sort": "recent",
            "page": page,
        }

        resp = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=15)
        if resp.status_code == 429:
            print("Rate limited — pausing 10 seconds...")
            time.sleep(10)
            continue
        resp.raise_for_status()

        jobs = resp.json().get("jobs", [])
        if not jobs:
            break  # No more pages for this query

        for job in jobs:
            guid = job.get("guid")
            if guid and guid not in jobs_by_guid:
                # Convert HTML description to clean plain text
                raw_html = job.get("description", "")
                clean_text = BeautifulSoup(raw_html, "html.parser").get_text(separator=" ", strip=True)
                job["clean_description"] = " ".join(clean_text.split())
                jobs_by_guid[guid] = job

        page += 1
        time.sleep(1)  # Polite 1-second delay between pages

df_himalayas = pd.DataFrame(jobs_by_guid.values())

# Filter to keep only jobs with explicit salary figures
df_himalayas = df_himalayas[df_himalayas["minSalary"].notna() | df_himalayas["maxSalary"].notna()].copy()

# Save to CSV
cols = [
    "guid", "title", "companyName", "seniority", "employmentType",
    "categories", "parentCategories", "minSalary", "maxSalary",
    "salaryPeriod", "currency", "clean_description"
]
df_himalayas[cols].to_csv("himalayas_data_science_jobs.csv", index=False)
print(f"Saved {len(df_himalayas)} unique Data Science jobs with salary data.")

Searching for: 'data scientist'...
Searching for: 'machine learning'...
Searching for: 'data engineer'...
Searching for: 'data analyst'...
Searching for: 'artificial intelligence'...
Searching for: 'applied scientist'...
Top parent categories: parentCategories
Data Science         3045
Developer            2871
Sales                 539
Operations            487
Finance               372
Product               291
Marketing             289
Hardware Engineer     221
Customer Service      190
Healthcare            147
Name: count, dtype: int64
Saved 4538 unique Data Science jobs with salary data.


In [ ]:
import numpy as np
import pandas as pd

# 1. Make an explicit, cleanly indexed copy before modifying columns
df_himalayas = df_himalayas.reset_index(drop=True)

# 2. Ensure salary columns are numeric
df_himalayas["minSalary"] = pd.to_numeric(df_himalayas["minSalary"], errors="coerce")
df_himalayas["maxSalary"] = pd.to_numeric(df_himalayas["maxSalary"], errors="coerce")

# 3. Calculate midpoint
df_himalayas["midpoint_salary"] = df_himalayas[["minSalary", "maxSalary"]].mean(axis=1)

# 4. Map salaryPeriod to annual multipliers
period_clean = df_himalayas["salaryPeriod"].fillna("unknown").str.lower().str.strip()

multipliers = {
    "annual": 1,
    "yearly": 1,
    "year": 1,
    "monthly": 12,
    "month": 12,
    "weekly": 52,
    "week": 52,
    "daily": 260,
    "day": 260,
    "hourly": 2080,
    "hour": 2080,
}

df_himalayas["multiplier"] = period_clean.map(multipliers)

# Fallback when salaryPeriod is missing
df_himalayas.loc[
    df_himalayas["multiplier"].isna() & (df_himalayas["midpoint_salary"] < 500),
    "multiplier",
] = 2080
df_himalayas.loc[
    df_himalayas["multiplier"].isna() & (df_himalayas["midpoint_salary"] >= 15000),
    "multiplier",
] = 1

# 5. Compute annualized salary
df_himalayas["annual_salary"] = (
    df_himalayas["midpoint_salary"] * df_himalayas["multiplier"])

In [ ]:
df_himalayas = df_himalayas[~df_himalayas['employmentType'].isin(['Contractor', 'Part Time', 'Temporary'])].reset_index()

In [ ]:
df_himalayas['employmentType'].value_counts()

,count
employmentType,
Full Time,3442
Other,37
Intern,23


In [ ]:
df_himalayas = df_himalayas.drop(columns=['excerpt', 'companyLogo', 'description', 'applicationLink', 'guid', 'multiplier'])

In [ ]:
df_himalayas = df_himalayas[df_himalayas['currency'] == 'USD'].reset_index()

In [ ]:
df_himalayas.to_csv('web_analytics_dataset.csv', index=False)

In [ ]:
df_himalayas = pd.read_csv('/content/drive/MyDrive/web_analytics_dataset.csv')

Combining the two datasets

In [ ]:
import re, ast
import numpy as np
import pandas as pd

# ---------- helpers ----------
def to_list(x):
    if isinstance(x, (list, tuple, set, np.ndarray)):
        items = list(x)
    elif isinstance(x, str):
        s = x.strip()
        try:
            v = ast.literal_eval(s)
            items = list(v) if isinstance(v, (list, tuple, set)) else [s]
        except Exception:
            items = s.split(",")
    else:
        return []
    out = []
    for i in items:
        if isinstance(i, dict):
            i = i.get("name", "")
        i = str(i).strip().lower()
        if i:
            out.append(i)
    return out

def norm_level(x):
    s = x.lower() if isinstance(x, str) else " ".join(to_list(x))
    if re.search(r"intern", s):                                                return "Intern"
    if re.search(r"entry|junior|\bjr\b|graduate", s):                          return "Entry"
    if re.search(r"\bmid", s):                                                 return "Mid"
    if re.search(r"senior|\bsr\b", s):                                         return "Senior"
    if re.search(r"lead|manager|director|executive|head|principal|\bvp\b", s): return "Lead+"
    return "Unknown"

HYBRID_RE = re.compile(
    r"hybrid (?:work|role|schedule|model|position|environment|setup|office)|"
    r"\bhybrid\b(?! cloud)|"
    r"\b[2-4]\s*(?:days?|x)\s*(?:per|a|/)\s*week\s*(?:in|at|on)[- ]?(?:the\s*)?(?:office|site)|"
    r"in[- ]office\s*[2-4]\s*days", re.I)
REMOTE_RE = re.compile(r"fully remote|100% remote|remote[- ]first|work from home|work remotely|this is a remote", re.I)

def muse_work_mode(location, description):
    loc, desc = str(location).lower(), str(description)
    if HYBRID_RE.search(desc):                 return "Hybrid"
    if "remote" in loc or "flexible" in loc:   return "Remote"
    if REMOTE_RE.search(desc):                 return "Remote"
    return "Onsite"

def norm_text(s):
    s = str(s).lower()
    s = re.sub(r"\(.*?\)|\[.*?\]", " ", s)
    s = re.sub(r"\b(remote|hybrid)\b", " ", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def parse_dates(s):
    num = pd.to_numeric(s, errors="coerce")
    a = pd.to_datetime(num.where(num.between(1e8, 1e11)),  unit="s",  utc=True)
    b = pd.to_datetime(num.where(num.between(1e11, 1e14)), unit="ms", utc=True)
    c = pd.to_datetime(s.where(num.isna()), errors="coerce", utc=True)
    return a.fillna(b).fillna(c)

# ---------- MUSE ----------
muse = pd.read_csv('/content/drive/MyDrive/muse_dataset_salaries_full (2).csv')
him = pd.read_csv('/content/drive/MyDrive/web_analytics_dataset.csv')

m = pd.DataFrame({
    "source": "muse",
    "job_id": muse["job_id"],
    "title": muse["title"],
    "company": muse["company"],
    "location": muse["locations"].map(lambda x: "; ".join(to_list(x))),
    "categories": muse["categories"].map(to_list),
    "level": muse["levels"].map(norm_level),
    "pub_date": parse_dates(muse["publication_date"]),
    "currency": muse["llm_currency"],
    "annual_salary": muse["llm_annual_salary"],
    "description": muse["description"],
    "url": muse["apply_url"],
})
m["work_mode"] = [muse_work_mode(l, d) for l, d in zip(m["location"], m["description"])]

# ---------- HIMALAYAS ----------
h = pd.DataFrame({
    "source": "himalayas",
    "job_id": np.nan,
    "title": him["title"],
    "company": him["companyName"],
    "location": him["locationRestrictions"].map(lambda x: "; ".join(to_list(x)) or "remote"),
    "categories": him["categories"].map(to_list),
    "level": him["seniority"].map(norm_level),
    "pub_date": parse_dates(him["pubDate"]),
    "currency": him["currency"],
    "annual_salary": him["annual_salary"],
    "description": him["clean_description"],
    "url": np.nan,
})
h["work_mode"] = "Remote"

# drop the exact same job_id
n0 = len(m)
m = m[m["job_id"].isna() | ~m["job_id"].duplicated(keep="first")]
print(f"Muse: removed {n0 - len(m)} rows with a repeated job_id (same posting)\n")
# Himalayas has no ID, so none of its rows are removed

before = pd.concat([m, h], ignore_index=True)
before["row_id"] = before.index
before["loose_key"] = (before["company"].map(norm_text) + "|" +
                       before["title"].map(norm_text) + "|" + before["level"])

# CROSS-SOURCE ONLY: Muse remote vs Himalayas
mu = before[(before["source"] == "muse") & (before["work_mode"] == "Remote")]
hi = before[before["source"] == "himalayas"]
pairs = mu.merge(hi, on="loose_key", suffixes=("_m", "_h"))

def salary_agrees(a, b):
    if pd.isna(a) or pd.isna(b):
        return True
    return abs(a - b) <= 0.01 * max(a, b)

pairs = pairs[[salary_agrees(a, b) for a, b in
               zip(pairs["annual_salary_m"], pairs["annual_salary_h"])]]

# pass the Muse salary to the Himalayas row if that one has none
fill = (pairs.dropna(subset=["annual_salary_m"])
             .drop_duplicates("row_id_h")
             .set_index("row_id_h")[["annual_salary_m", "currency_m"]])
mask = before["row_id"].isin(fill.index) & before["annual_salary"].isna()
before.loc[mask, "annual_salary"] = before.loc[mask, "row_id"].map(fill["annual_salary_m"])
before.loc[mask, "currency"]      = before.loc[mask, "row_id"].map(fill["currency_m"])

drop_ids = set(pairs["row_id_m"])
df = before[~before["row_id"].isin(drop_ids)].reset_index(drop=True)
df["job_uid"] = df.index

print(f"Cross-source matches: {len(pairs)} pairs -> dropped {len(drop_ids)} Muse rows")
print(f"Final: {len(before)} -> {len(df)} rows\n")
df.to_csv("combined_data.csv")

In [ ]:
combined_data = pd.read_csv('/content/drive/MyDrive/combined_data.csv')

In [ ]:
import os
import re
import json
import time
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

API_KEY = userdata.get("LUXIA_API_KEY")

LUXIA_ENDPOINT = "https://bridge.luxiacloud.com/luxia/v1/chat"
MODEL_NAME = "luxia3-llm-8b-0731"

CANONICAL_SKILLS = [
    "python", "sql", "r", "c++", "c#", "java", "javascript", "typescript",
    "go", "rust", "scala", "bash/shell",
    "pytorch", "tensorflow", "keras", "scikit-learn", "pandas", "numpy",
    "apache spark", "hadoop", "airflow", "databricks", "langchain",
    "hugging face", "llm / genai", "computer vision", "nlp",
    "aws", "azure", "gcp", "docker", "kubernetes", "terraform",
    "linux", "git", "ci/cd",
    "postgresql", "mysql", "mongodb", "redis", "snowflake", "bigquery", "redshift",
    "tableau", "power bi", "looker",
]
ALLOWED_SET = set(CANONICAL_SKILLS)

ALIASES = {
    "golang": "go", "go language": "go", "r programming": "r", "rstudio": "r",
    "cpp": "c++", "csharp": "c#", "js": "javascript", "ts": "typescript",
    "bash": "bash/shell", "shell": "bash/shell", "sklearn": "scikit-learn",
    "spark": "apache spark", "pyspark": "apache spark", "huggingface": "hugging face",
    "llm": "llm / genai", "llms": "llm / genai", "genai": "llm / genai",
    "generative ai": "llm / genai", "natural language processing": "nlp",
    "amazon web services": "aws", "google cloud": "gcp", "google cloud platform": "gcp",
    "k8s": "kubernetes", "github": "git", "gitlab": "git", "cicd": "ci/cd",
    "postgres": "postgresql", "mongo": "mongodb", "powerbi": "power bi",
}

# Prompt focused strictly on skill extraction—no title-based role rejection
SYSTEM_PROMPT = f"""You are an expert NLP data extraction system specializing in parsing technical skills from unstructured job descriptions.

Your objective: Extract all technical skills mentioned in the job posting (even if the job title sounds non-traditional, such as Operations, Finance, Strategy, or Research) mapped to our canonical vocabulary.
Return ONLY valid JSON matching this exact structure with no Markdown wrappers, commentary, or explanations:

{{
  "skills": [string] // list of matched skills chosen ONLY from the allowed list below
}}

Allowed canonical skills:
{json.dumps(CANONICAL_SKILLS)}

Strict extraction rules:
- Include "go" ONLY for the Go / Golang programming language (NEVER the English verb "to go").
- Include "r" ONLY for the R statistical programming language / RStudio (NEVER "R&D" or standalone letters).
- Normalize synonyms (e.g., PySpark -> "apache spark", k8s -> "kubernetes", Postgres -> "postgresql", GenAI/LLMs -> "llm / genai").
- If no allowed technical skills are mentioned in the text, return {{"skills": []}}.
"""

session = requests.Session()
session.headers.update({"apikey": API_KEY, "Content-Type": "application/json"})

EMPTY = {"is_tech_or_data_role": False, "skills": []}


def normalize_skills(raw_skills) -> list[str]:
    if not isinstance(raw_skills, list):
        return []
    cleaned = set()
    for item in raw_skills:
        s = ALIASES.get(str(item).strip().lower(), str(item).strip().lower())
        if s in ALLOWED_SET:
            cleaned.add(s)
    return sorted(cleaned)


def extract_skills_with_luxia(title: str, job_text: str, retries: int = 3) -> dict:
    if not isinstance(job_text, str) or not job_text.strip():
        return EMPTY.copy()

    user_content = f"Job Title: {title}\n\nJob Description:\n{job_text}"

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "temperature": 0.0,
        "max_tokens": 300,
    }

    for attempt in range(retries):
        try:
            r = session.post(LUXIA_ENDPOINT, json=payload, timeout=45)
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            content = r.json()["choices"][0]["message"]["content"].strip()
            if content.startswith("```"):
                content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content, flags=re.IGNORECASE).strip()
            parsed = json.loads(content)
            return {
                "is_tech_or_data_role": bool(parsed.get("is_tech_or_data_role", False)),
                "skills": normalize_skills(parsed.get("skills", [])),
            }
        except Exception as e:
            if attempt == retries - 1:
                print(f"Extraction error ({title[:40]}): {e}")
            else:
                time.sleep(1 + attempt)
    return EMPTY.copy()


def run_skill_extraction(df: pd.DataFrame, workers: int = 7) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Deduplicate identical (title, description) pairs so duplicate postings are only sent once
    pairs_df = df[["title", "description"]].fillna("").drop_duplicates()
    unique_items = list(pairs_df.itertuples(index=False, name=None))
    results = {}

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {
            ex.submit(extract_skills_with_luxia, title, desc): (title, desc)
            for title, desc in unique_items
        }
        for f in tqdm(as_completed(futures), total=len(futures), desc="Extracting skills via LUXIA"):
            results[futures[f]] = f.result()

    # Map results back to every row in df
    keys = list(zip(df["title"].fillna(""), df["description"].fillna("")))
    extracted = [results.get(k, EMPTY.copy()) for k in keys]

    out = df.copy()
    out["skills_clean"] = [e["skills"] for e in extracted]
    out["skills"] = [str(e["skills"]) for e in extracted]

    # Filter strictly to verified tech/data roles with >= 1 valid skill
    clean_jobs = out[(out["skills_clean"].map(len) > 0)].copy()

    return clean_jobs


df = pd.read_csv("/content/drive/MyDrive/combined_data.csv")
clean_jobs_df = run_skill_extraction(df, workers=7)
clean_jobs_df.to_csv("combined_data_tech_only.csv", index=False)
print(f"Kept {len(clean_jobs_df):,} verified tech/data jobs out of {len(df):,} ({len(clean_skills_df):,} skill pairs)")